# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation to our models using distributed processing. Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a small model (student) is trained to mimic a larger, more complex model (teacher). The key insight is that the teacher model's outputs contain rich information beyond just the predicted class - they contain the relative probabilities across all classes, which represent the teacher's "dark knowledge".

### How Knowledge Distillation Works:

1. **Teacher Model**: A large, pre-trained model with high accuracy but high computational requirements
2. **Student Model**: A smaller model architecture that we want to train
3. **Distillation Process**: The student is trained using a combination of:
   - **Hard Targets**: The actual ground truth labels (standard supervised learning)
   - **Soft Targets**: The probability distributions output by the teacher model

### Benefits of Knowledge Distillation:
- **Reduced Model Size**: Student models are typically much smaller than teacher models
- **Faster Inference**: Smaller models require less computation for predictions
- **Lower Memory Requirements**: Smaller models use less memory during inference
- **Preserved Accuracy**: Student models often retain much of the teacher's performance

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

## 3. Load Model Information

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Define Student Model Architectures

For each teacher model, we need to define a smaller student model architecture. We'll use pre-defined architectures that are known to work well as student models.

In [ ]:
# Define student model architectures for each teacher model
student_architectures = {
    "sentiment-analysis": {
        "model_name": "distilbert-base-uncased",  # Use DistilBERT as student for BERT-based models
        "num_hidden_layers": 3,  # Reduce the number of layers for even smaller model
        "hidden_size": 384  # Reduce hidden size for smaller model
    },
    "ner": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 3,
        "hidden_size": 384
    },
    "question-answering": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 4,  # Slightly more layers for QA task
        "hidden_size": 384
    },
    "masked-lm": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 3,
        "hidden_size": 384
    }
}

# Print student architectures
for model_key, architecture in student_architectures.items():
    if model_key in model_info:
        print(f"Student architecture for {model_key}:")
        print(f"  Base model: {architecture['model_name']}")
        print(f"  Hidden layers: {architecture['num_hidden_layers']}")
        print(f"  Hidden size: {architecture['hidden_size']}")
        print()

## 5. Create Distillation Script

In this section, we'll create a Python script that performs the actual knowledge distillation. This script will be executed on the SageMaker Processing instances.

In [ ]:
# Check if the distillation_scripts directory exists, if not create it
import os
if not os.path.exists('distillation_scripts'):
    os.makedirs('distillation_scripts')
    print("Created distillation_scripts directory")
else:
    print("distillation_scripts directory already exists")

## 6. Launch Distributed Distillation Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform knowledge distillation. Each model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Define the instance type to use for distillation
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="knowledge-distillation",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch distillation jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing distillation jobs for all models...")

for model_key in model_info.keys():
    # Check if we have a student architecture for this model
    if model_key not in student_architectures:
        print(f"No student architecture defined for {model_key}, skipping...")
        continue
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Save student architecture to a temporary file
    with open(f'temp_{model_key}_student.json', 'w') as f:
        json.dump({model_key: student_architectures[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    s3_client.upload_file(
        f'temp_{model_key}_student.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/student_architecture.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-distilled'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/teacher'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/student_architecture.json',
            destination='/opt/ml/processing/input/student'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='distilled-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--teacher-info-path', '/opt/ml/processing/input/teacher/model_info.json',
            '--student-info-path', '/opt/ml/processing/input/student/student_architecture.json',
            '--output-dir', '/opt/ml/processing/output',
            '--temperature', '2.0',  # Temperature for softening the teacher's outputs
            '--alpha', '0.5',  # Weight for distillation loss vs. task loss
            '--epochs', '3',  # Number of training epochs
            '--batch-size', '16'  # Batch size for training
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all distillation jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"distillation-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='distillation_script.py',
            source_dir='distillation_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 7. Collect Results

Now that the distillation jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the distilled model, such as size, inference time, and the comparison with the teacher model.

### Collection Process:
1. **Download metrics files** from S3 for each model
2. **Combine metrics** into a single dictionary
3. **Save combined metrics** to a local file for use in later notebooks

In [ ]:
# Download and combine results
distilled_metrics = {}
sagemaker_client = boto3.client("sagemaker")

# Debug: Print job output paths
print(f"Job output paths: {job_output_paths}")

for model_key in model_info.keys():
    # Check if we have a job for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path
        metrics_path = f'optimization/outputs/{model_key}-distilled/distilled-metrics.json'
        local_path = f'temp_{model_key}-distilled-metrics.json'
        
        print(f"Attempting to download from s3://{S3_BUCKET}/{metrics_path} to {local_path}")
        
        s3_client.download_file(
            S3_BUCKET,
            metrics_path,
            local_path
        )
        
        # Load metrics
        with open(local_path, 'r') as f:
            metrics = json.load(f)
            print(f"Loaded metrics for {model_key}: {list(metrics.keys())}")
        
        # Add to combined metrics
        distilled_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")
        print(f"Checking if file exists in S3...")
        try:
            response = s3_client.list_objects_v2(
                Bucket=S3_BUCKET,
                Prefix=f'optimization/outputs/{model_key}-distilled/'
            )
            if 'Contents' in response:
                print(f"Files found in S3 for {model_key}:")
                for obj in response['Contents']:
                    print(f"  - {obj['Key']}")
            else:
                print(f"No files found in S3 for {model_key}")
        except Exception as e2:
            print(f"Error checking S3: {e2}")

# Save combined metrics
with open('distilled-metrics.json', 'w') as f:
    json.dump(distilled_metrics, f, indent=2)

print(f"\nSaved distilled metrics for {len(distilled_metrics)} models to distilled-metrics.json")
print(f"Metrics keys: {list(distilled_metrics.keys()) if distilled_metrics else 'No metrics available'}")


## 8. Analyze Model Size Reduction

Now we'll analyze the size reduction achieved through knowledge distillation. This analysis helps us understand the impact of distillation on model size and memory footprint.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

for model_key in distilled_metrics.keys():
    distilled = distilled_metrics[model_key]
    
    # Get original model size from S3
    original_model_prefix = model_info[model_key]['s3_uri'].replace(f"s3://{S3_BUCKET}/", "")
    original_size = get_total_size(S3_BUCKET, original_model_prefix)
    original_size_mb = original_size / (1024 * 1024)
    
    # Get teacher and student model sizes
    teacher_size_mb = distilled['teacher']['model_size']
    student_size_mb = distilled['student']['model_size']
    
    # Calculate size reduction percentage
    size_reduction = ((teacher_size_mb - student_size_mb) / teacher_size_mb) * 100 if teacher_size_mb > 0 else 0
    
    # Prepare data for this model
    model_data = {
        'Model': distilled['teacher']['model_name'],
        'Teacher Size (MB)': f"{teacher_size_mb:.2f}",
        'Student Size (MB)': f"{student_size_mb:.2f}",
        'Size Reduction (%)': f"{size_reduction:.2f}",
        'Teacher Accuracy (%)': f"{distilled['teacher'].get('accuracy', 0) * 100:.2f}",
        'Student Accuracy (%)': f"{distilled['student'].get('accuracy', 0) * 100:.2f}",
        'Accuracy Retention (%)': f"{(distilled['student'].get('accuracy', 0) / distilled['teacher'].get('accuracy', 1)) * 100:.2f}"
    }
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Visualize Inference Speed Comparison

Let's visualize the inference speed improvement achieved through knowledge distillation.

In [ ]:
# Create a DataFrame for visualization
viz_data = []

for model_key in distilled_metrics.keys():
    # Extract model name in a more reliable way
    if 'teacher' in distilled_metrics[model_key] and 'model_name' in distilled_metrics[model_key]['teacher']:
        model_name = distilled_metrics[model_key]['teacher']['model_name'].split('/')[-1]  # Get just the model name without path
        
        # Get inference times
        teacher_time = distilled_metrics[model_key]['teacher'].get('inference_time', 0)
        student_time = distilled_metrics[model_key]['student'].get('inference_time', 0)
        
        # Get model sizes
        teacher_size = distilled_metrics[model_key]['teacher'].get('model_size', 0)
        student_size = distilled_metrics[model_key]['student'].get('model_size', 0)
        
        # Add teacher data
        viz_data.append({
            'Model': model_name,
            'Type': 'Teacher',
            'Inference Time (ms)': teacher_time,
            'Model Size (MB)': teacher_size
        })
        
        # Add student data
        viz_data.append({
            'Model': model_name,
            'Type': 'Student',
            'Inference Time (ms)': student_time,
            'Model Size (MB)': student_size
        })

# Create DataFrame for visualization
viz_df = pd.DataFrame(viz_data)

# Check if the DataFrame has data
if not viz_df.empty:
    print(f"\nVisualization DataFrame has {len(viz_df)} rows")
    print(viz_df.head())
    
    # Create the inference time bar chart
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x="Model", y="Inference Time (ms)", hue="Type", data=viz_df)
    
    # Customize the chart
    plt.title("Inference Time Comparison: Teacher vs. Student Models")
    plt.xlabel("Model")
    plt.ylabel("Inference Time (ms)")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Model Type")
    plt.tight_layout()
    
    # Show the chart
    plt.show()
    
    # Create the model size bar chart
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x="Model", y="Model Size (MB)", hue="Type", data=viz_df)
    
    # Customize the chart
    plt.title("Model Size Comparison: Teacher vs. Student Models")
    plt.xlabel("Model")
    plt.ylabel("Model Size (MB)")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Model Type")
    plt.tight_layout()
    
    # Show the chart
    plt.show()
else:
    print("No data available for visualization. Make sure distillation jobs completed successfully.")


## 10. Next Steps

Now that we've applied knowledge distillation to our models and analyzed the results, we'll explore cost analysis in the next notebook to understand the financial implications of these optimization techniques.

### What We've Learned:
- How knowledge distillation works by transferring knowledge from a teacher model to a student model
- How to implement knowledge distillation using SageMaker Processing jobs
- How distillation affects model size, inference time, and accuracy
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - Cost Analysis:
In the next notebook, we'll analyze the cost implications of the various optimization techniques we've explored, including quantization, pruning, and knowledge distillation. We'll calculate the ROI and payback period for each technique to help you make informed decisions about which techniques to apply in your own projects.